In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp03/ex_3.py ──
"""
Shared infrastructure for MLFP03 Exercise 3 — The Classical ML Zoo.

Contains: e-commerce churn data loading, preprocessing, CV strategy,
2D PCA projection for decision boundary plots, model comparison helpers,
and a shared ModelVisualizer-backed plot utility.

Technique-specific code (model fitting, parameter sweeps, from-scratch
Gini, OOB convergence, decision guide) does NOT belong here — it lives
in the per-technique files under modules/mlfp03/solutions/ex_3/.
"""

import time
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score

from kailash_ml import ModelVisualizer, PreprocessingPipeline
from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT
# ════════════════════════════════════════════════════════════════════════

setup_environment()
np.random.seed(42)

# Output directory for comparison artifacts (HTML plots, tables)
OUTPUT_DIR = Path("outputs") / "ex3_model_zoo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# E-commerce churn dataset — Singapore APAC retail churn scenario
DATASET_MODULE = "mlfp03"
DATASET_FILE = "ecommerce_customers.parquet"
TARGET_COL = "churned"

# SVM with RBF kernel is O(n²) — cap the training set so every technique
# in the zoo fits in a few seconds on a laptop.
SUBSAMPLE_N = 5000
RANDOM_SEED = 42

# Drop columns that are text/ID or leak the target
DROP_COLS = ["customer_id", "review_text", "product_categories"]


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING + PREPROCESSING
# ════════════════════════════════════════════════════════════════════════


def load_ecommerce_churn() -> pl.DataFrame:
    """Load the Singapore e-commerce churn dataset (polars DataFrame).

    Drops text/ID columns and subsamples for SVM tractability.
    """
    loader = MLFPDataLoader()
    df = loader.load(DATASET_MODULE, DATASET_FILE)
    df = df.sample(n=min(SUBSAMPLE_N, df.height), seed=RANDOM_SEED)
    keep = [c for c in df.columns if c not in DROP_COLS]
    return df.select(keep)


def build_train_test_split() -> dict[str, Any]:
    """Return a fully prepared dict: X_train, X_test, y_train, y_test, feature_names, cv.

    Uses kailash_ml PreprocessingPipeline with z-score normalisation and
    ordinal categorical encoding. Every technique file calls this so all
    models share identical folds and identical preprocessing.
    """
    df = load_ecommerce_churn()

    pipeline = PreprocessingPipeline()
    result = pipeline.setup(
        data=df,
        target=TARGET_COL,
        train_size=0.8,
        seed=RANDOM_SEED,
        normalize=True,
        normalize_method="zscore",
        categorical_encoding="ordinal",
        imputation_strategy="median",
    )

    feature_cols = [c for c in result.train_data.columns if c != TARGET_COL]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_cols,
        target_column=TARGET_COL,
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_cols,
        target_column=TARGET_COL,
    )
    feature_names = col_info["feature_columns"]

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

    return {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "feature_names": feature_names,
        "cv": cv,
        "churn_rate": float(np.mean(y_train)),
    }


# ════════════════════════════════════════════════════════════════════════
# 2D PCA PROJECTION — shared so every technique plots on the same axes
# ════════════════════════════════════════════════════════════════════════


def project_2d(X_train: np.ndarray, X_test: np.ndarray) -> dict[str, Any]:
    """Fit PCA(2) on X_train and project both train and test.

    Returns {X_train_2d, X_test_2d, explained_variance, pca}.
    """
    pca = PCA(n_components=2, random_state=RANDOM_SEED)
    X_train_2d = pca.fit_transform(X_train)
    X_test_2d = pca.transform(X_test)
    return {
        "X_train_2d": X_train_2d,
        "X_test_2d": X_test_2d,
        "explained_variance": pca.explained_variance_ratio_,
        "pca": pca,
    }


# ════════════════════════════════════════════════════════════════════════
# CROSS-VALIDATION HELPER — keep every parameter sweep one line
# ════════════════════════════════════════════════════════════════════════


def cv_accuracy_f1(
    estimator: Any,
    X: np.ndarray,
    y: np.ndarray,
    cv: Any,
) -> tuple[float, float]:
    """Return (mean_accuracy, mean_f1) for a 5-fold CV."""
    acc = cross_val_score(estimator, X, y, cv=cv, scoring="accuracy").mean()
    f1 = cross_val_score(estimator, X, y, cv=cv, scoring="f1").mean()
    return float(acc), float(f1)


# ════════════════════════════════════════════════════════════════════════
# EVALUATION — train on full set, measure timing, return pred/prob/metrics
# ════════════════════════════════════════════════════════════════════════


def fit_and_evaluate(
    estimator: Any,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    name: str,
) -> dict[str, Any]:
    """Fit, predict, score, and time a single model.

    Returns a dict with keys: name, model, pred, prob, train_time,
    accuracy, f1, auc_roc.
    """
    t0 = time.perf_counter()
    estimator.fit(X_train, y_train)
    train_time = time.perf_counter() - t0

    pred = estimator.predict(X_test)
    if hasattr(estimator, "predict_proba"):
        prob = estimator.predict_proba(X_test)[:, 1]
    else:
        # Decision-function fallback (never used by the zoo but keeps contract)
        prob = estimator.decision_function(X_test)

    return {
        "name": name,
        "model": estimator,
        "pred": pred,
        "prob": prob,
        "train_time": float(train_time),
        "accuracy": float(accuracy_score(y_test, pred)),
        "f1": float(f1_score(y_test, pred)),
        "auc_roc": float(roc_auc_score(y_test, prob)),
    }


def print_classification_report(y_test: np.ndarray, pred: np.ndarray) -> None:
    """Print sklearn classification report with churn-friendly target names."""
    print(
        classification_report(
            y_test,
            pred,
            target_names=["Retained", "Churned"],
        )
    )


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION — Plotly via kailash_ml.ModelVisualizer
# ════════════════════════════════════════════════════════════════════════


def get_visualizer() -> ModelVisualizer:
    """Return a ModelVisualizer instance (polars-native plots)."""
    return ModelVisualizer()


def save_metric_comparison(
    metric_dict: dict[str, dict[str, float]], fname: str
) -> Path:
    """Save a metric_comparison plot to OUTPUT_DIR/fname and return the path."""
    viz = get_visualizer()
    fig = viz.metric_comparison(metric_dict)
    fig.update_layout(title="Classical ML Zoo — Performance Comparison")
    out = OUTPUT_DIR / fname
    fig.write_html(str(out))
    return out


# ════════════════════════════════════════════════════════════════════════
# DECISION BOUNDARY MESH — shared helper so every technique file uses
# the same axes, grid, and figure style.
# ════════════════════════════════════════════════════════════════════════


def decision_boundary_mesh(
    X_2d: np.ndarray,
    step: float = 0.1,
    pad: float = 0.5,
) -> tuple[np.ndarray, np.ndarray]:
    """Return (xx, yy) meshgrid covering the 2D PCA projection."""
    x_min, x_max = X_2d[:, 0].min() - pad, X_2d[:, 0].max() + pad
    y_min, y_max = X_2d[:, 1].min() - pad, X_2d[:, 1].max() + pad
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, step),
        np.arange(y_min, y_max, step),
    )
    return xx, yy


# ════════════════════════════════════════════════════════════════════════
# SINGAPORE E-COMMERCE CHURN — business-impact constants
# ════════════════════════════════════════════════════════════════════════
# Public industry figures used for the "Apply" phases. Sources in reading
# notes (SGX retail analyst reports, Shopee/Lazada 2024 ops reviews).

AVG_CUSTOMER_LIFETIME_VALUE_SGD = 420.0  # avg 12-month CLV per retained SG customer
RETENTION_OFFER_COST_SGD = 18.0  # targeted promo cost per flagged customer
MONTHLY_ACTIVE_CUSTOMERS = 250_000  # typical mid-market SG e-commerce platform
ANNUAL_CHURN_BASELINE = 0.22  # industry baseline annual churn


def churn_saved_dollars(true_positives: int) -> float:
    """Dollar value of correctly identified churners (retention offer accepted).

    Assumes a 40% offer-acceptance rate and the retained lifetime value
    net of offer cost. Public industry benchmarks — not proprietary data.
    """
    accept_rate = 0.40
    net_value_per_save = AVG_CUSTOMER_LIFETIME_VALUE_SGD - RETENTION_OFFER_COST_SGD
    return round(true_positives * accept_rate * net_value_per_save, 2)




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 3.1: Support Vector Machines (Linear + RBF)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Understand margin maximisation and the soft-margin C parameter
#   - Train a linear SVM and an RBF-kernel SVM with sklearn
#   - Sweep C across orders of magnitude and pick the best via CV
#   - Visualise the RBF decision boundary in 2D PCA space
#   - Translate SVM accuracy into Singapore retail churn dollars saved
#
# PREREQUISITES: MLFP03 Exercise 2 (bias-variance, regularisation)
#
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Theory — margin maximisation and the kernel trick
#   2. Build — linear + RBF SVM, C parameter sweep
#   3. Train — final RBF SVM on the full training set
#   4. Visualise — 2D decision boundary, support vector overlay
#   5. Apply — Singapore e-commerce churn cost-benefit
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
from dotenv import load_dotenv
from sklearn.svm import SVC


load_dotenv()



## THEORY — Margin Maximisation and the Kernel Trick


In [ ]:
# An SVM finds the hyperplane that separates two classes with the LARGEST
# possible gap (margin) between them. Intuition: a wider gap means any
# small perturbation in the data is less likely to flip a prediction.
#
#   Hard-margin primal:  minimise ||w||² / 2
#                        subject to y_i (w . x_i + b) >= 1 for all i
#
#   Soft-margin: add slack variables ξ_i that let the SVM misclassify
#   points at a cost C per unit of slack.
#     C -> infinity  : hard margin, no misclassification tolerated
#     C -> 0         : very wide margin, many misclassifications tolerated
#
# The KERNEL TRICK lets the SVM learn a nonlinear boundary without ever
# materialising the high-dimensional feature map φ(x). It computes inner
# products in that space via a kernel K(x, x'):
#     Linear : K(x, x') = x . x'
#     RBF    : K(x, x') = exp(-γ ||x - x'||²)
#
# RBF is the default choice for tabular data because it can bend the
# decision surface around arbitrary clusters.



## TASK 2 — BUILD: Load data, sweep C for linear and RBF kernels


Sweep C for a given kernel and return {C: {accuracy, f1}}.


In [ ]:
print("\n" + "=" * 70)
print("  MLFP03 Exercise 3.1 — Support Vector Machines")
print("=" * 70)

data = build_train_test_split()
X_train, X_test = data["X_train"], data["X_test"]
y_train, y_test = data["y_train"], data["y_test"]
cv = data["cv"]
feature_names = data["feature_names"]

print(f"\nTrain: {X_train.shape}, Test: {X_test.shape}")
print(f"Churn rate (train): {data['churn_rate']:.2%}")

C_VALUES = [0.01, 0.1, 1.0, 10.0, 100.0]


def sweep_c(kernel: str) -> dict[float, dict[str, float]]:
    results: dict[float, dict[str, float]] = {}
    print(f"\n--- {kernel.upper()} SVM: C parameter sweep ---")
    print(f"{'C':>10} {'CV Accuracy':>14} {'CV F1':>10}")
    print("-" * 38)
    for c_val in C_VALUES:
        acc, f1 = cv_accuracy_f1(
            SVC(kernel=kernel, C=c_val, random_state=RANDOM_SEED),
            X_train,
            y_train,
            cv,
        )
        results[c_val] = {"accuracy": acc, "f1": f1}
        print(f"{c_val:>10.2f} {acc:>14.4f} {f1:>10.4f}")
    return results


linear_results = sweep_c("linear")
rbf_results = sweep_c("rbf")

best_c_linear = max(linear_results, key=lambda c: linear_results[c]["f1"])
best_c_rbf = max(rbf_results, key=lambda c: rbf_results[c]["f1"])
print(f"\nBest C — linear: {best_c_linear}, RBF: {best_c_rbf}")



## TASK 3 — TRAIN: final RBF SVM on the full training set


In [ ]:
svm_result = fit_and_evaluate(
    SVC(kernel="rbf", C=best_c_rbf, random_state=RANDOM_SEED, probability=True),
    X_train,
    y_train,
    X_test,
    y_test,
    name=f"SVM (RBF, C={best_c_rbf})",
)

print(
    f"\n{svm_result['name']}: trained in {svm_result['train_time']:.2f}s | "
    f"accuracy={svm_result['accuracy']:.4f} | "
    f"F1={svm_result['f1']:.4f} | AUC={svm_result['auc_roc']:.4f}"
)
print_classification_report(y_test, svm_result["pred"])

svm_model = svm_result["model"]
n_sv = int(svm_model.support_vectors_.shape[0])
sv_pct = n_sv / len(y_train)
print(
    f"Support vectors: {n_sv} ({sv_pct:.1%} of training). "
    f"A healthy SVM uses 10-30% of training points."
)



### Checkpoint 1


In [ ]:
assert svm_result["accuracy"] > 0.5, "SVM must beat random"
assert best_c_rbf in C_VALUES, "Best C must come from the sweep"
print("\n[ok] Checkpoint 1 passed — SVM trained and evaluated\n")



## TASK 4 — VISUALISE: 2D decision boundary + C-sweep curve


In [ ]:
pca_bundle = project_2d(X_train, X_test)
X_train_2d = pca_bundle["X_train_2d"]

svm_2d = SVC(kernel="rbf", C=best_c_rbf, random_state=RANDOM_SEED)
svm_2d.fit(X_train_2d, y_train)

xx, yy = decision_boundary_mesh(X_train_2d)
Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

viz = get_visualizer()
# C-sweep as a training-history style plot (accuracy and F1 vs log C)
history = {
    "linear accuracy": [linear_results[c]["accuracy"] for c in C_VALUES],
    "linear F1": [linear_results[c]["f1"] for c in C_VALUES],
    "rbf accuracy": [rbf_results[c]["accuracy"] for c in C_VALUES],
    "rbf F1": [rbf_results[c]["f1"] for c in C_VALUES],
}
fig_sweep = viz.training_history(history, x_label="C value (index)")
fig_sweep.update_layout(title="SVM: linear vs RBF C sweep (CV accuracy / F1)")
sweep_out = OUTPUT_DIR / "ex3_01_svm_c_sweep.html"
fig_sweep.write_html(str(sweep_out))
print(f"Saved: {sweep_out}")

# Decision boundary: export the mesh + decision so the notebook can
# render either via ModelVisualizer or a static matplotlib preview.
print(
    f"Decision mesh shape: {Z.shape} | "
    f"PCA variance captured: {pca_bundle['explained_variance'].sum():.2%}"
)



### Checkpoint 2


In [ ]:
assert Z.shape[0] > 0 and Z.shape[1] > 0, "Decision boundary mesh is empty"
print("[ok] Checkpoint 2 passed — 2D decision boundary computed\n")



## TASK 5 — APPLY: Singapore e-commerce churn cost-benefit


In [ ]:
# SCENARIO: A mid-market Singapore e-commerce marketplace (comparable to
# the SG-listed Shopee/Lazada footprint) has ~250K active customers per
# month. Annual churn baseline is ~22%. The retention team runs a
# targeted promo campaign (S$18 offer) against customers the model
# flags as likely churners.
#
# Why SVM is a reasonable candidate:
#   - Tabular feature space has around 20-30 behavioural features.
#     RBF SVM handles mid-dimensional data very well.
#   - The margin-based decision rule gives a calibrated "how close to
#     the boundary" signal that can prioritise the highest-risk flags.
#   - Training cost is a one-off nightly batch job — O(n²) is tolerable
#     at the subsampled 5K training size.
#
# LIMITATIONS:
#   - RBF SVM is a black box — the retention team cannot explain WHY a
#     customer was flagged. Compliance teams (PDPA, MAS) prefer
#     interpretable models for automated decisions.
#   - Scaling beyond ~50K training samples makes the O(n²) kernel matrix
#     impractical. For the full 250K customer base, move to Random
#     Forest or gradient boosting.

true_positives = int(((svm_result["pred"] == 1) & (y_test == 1)).sum())
dollars_saved = churn_saved_dollars(true_positives)
print(f"\nBusiness impact on held-out test set ({len(y_test)} customers):")
print(f"  True positives (churners caught): {true_positives}")
print(f"  Net retention value at 40% offer acceptance: S${dollars_saved:,.2f}")
print(
    f"  Extrapolated to the 250K monthly active base "
    f"(identical churn rate): "
    f"S${dollars_saved * (250_000 / len(y_test)):,.0f} / month"
)



## REFLECTION


[x] Margin maximisation and the C parameter trade-off
  [x] Linear vs RBF kernels and when each is appropriate
  [x] CV-driven C selection across five orders of magnitude
  [x] Held-out accuracy: {svm_result['accuracy']:.4f}, F1: {svm_result['f1']:.4f}
  [x] 2D PCA decision boundary for visual intuition
  [x] Translated classifier output into S${dollars_saved:,.0f} of retained
      customer value on the held-out test fold

  Next: 02_knn.py — instance-based learning with no training phase.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

